# The test dataset:

https://demo.borealisdata.ca/dataset.xhtml?persistentId=doi:10.80240/FK2/WEYHSP

In [2]:
import os
import sys
import yaml
import pandas as pd
import datetime
import janitor
import hashlib
import pydatacuration.utils as utils

In [ ]:
# Load the configuration file
with open('config.yaml', 'r') as f:
    config = yaml.load(f, Loader=yaml.FullLoader)

In [ ]:
# Redirect Tree structure to a text file
original_stdout = sys.stdout # Save a reference to the original standard output
with open('./log/ds_structure.txt', 'w') as f:
    sys.stdout = f # Change the standard output to the file we created.
    utils.list_files('data')
    sys.stdout = original_stdout # Reset the standard output to its original value

In [ ]:
# Export the structure ('tree') of a directory and its files as a dictionary
def get_filepaths(directory):
    file_info = {}
    id = 1  # Initialize id outside the loop
    # Walk the tree
    for root, dir, files in os.walk(directory):
        for file in files:
            full_file_path = os.path.join(root, file)
            parent_directory = os.path.basename(os.path.dirname(full_file_path))
            file_info[id] = {  # Use id as a key in file_info
                'root': directory,
                'parent_directory': parent_directory,
                'depth': full_file_path.count(os.sep) - directory.count(os.sep) + 1,
                'file': file,
                'file_path': full_file_path
            }
            id += 1  # Increment id after adding the file info to the dictionary
    return file_info

In [ ]:
def sha256sum(filepath):
    with open(filepath, 'rb', buffering=0) as f:
        return hashlib.file_digest(f, 'sha256').hexdigest()

In [ ]:
df = pd.DataFrame.from_dict(get_filepaths('data'), orient='index')

In [ ]:
# Update existing DataFrame in place
for index, row in df.iterrows():
    file_path = row['file_path']
    if os.path.exists(file_path):
        size = int(os.path.getsize(file_path))
        created = datetime.datetime.fromtimestamp(os.path.getctime(file_path))
        modified = datetime.datetime.fromtimestamp(os.path.getmtime(file_path))
        file_extension = os.path.splitext(file_path)[1]
        sha256_hash = sha256sum(file_path)
        df.at[index, 'file_id'] = index
        df.at[index, 'size'] = size
        df.at[index, 'created'] = created
        df.at[index, 'modified'] = modified
        df.at[index, 'file_extension'] = file_extension
        df.at[index, 'sha256_hash'] = sha256_hash
    else:
        df.at[index, 'size'] = None
        df.at[index, 'created'] = None
        df.at[index, 'modified'] = None

# Create a new DataFrame by copying the updated one
new_df = df.copy()
new_df = new_df.reorder_columns(['file_id', 'root'])

In [ ]:
new_df.to_csv('./log/ds_file_info.csv', index=False)

In [ ]:
# Initialize an empty list to collect file details
file_details = []


for file in get_filepaths('data'):
    size = os.path.getsize(file)
    created = datetime.datetime.fromtimestamp(os.path.getctime(file))
    modified = datetime.datetime.fromtimestamp(os.path.getmtime(file))
    file_details.append({'filename': file, 'size': size, 'created': created, 'modified': modified})

# Create DataFrame from the list
df_file_details = pd.DataFrame(file_details)

# Save the DataFrame to a CSV file
df.to_csv('file_management_log.csv', index=False)

# Interact with the remote kernel

In [ ]:
import subprocess

# Run the pip install command
#subprocess.run(['pip', 'install', 'opf-fido'], capture_output=True, text=True)
#print(subprocess.run(['pip', 'install', 'python-magic'], capture_output=True, text=True))